# Exercise MP3

## Exercise 1: Machine Epsilon

Find machine epsilon experimentally by successive halving and compare it to `np.finfo(dtype).eps`.


In [1]:
import numpy as np

def find_machine_epsilon(dtype=np.float64):
    eps = dtype(1.0)

    while dtype(1.0) + eps / dtype(2.0) != dtype(1.0):
        eps = eps / dtype(2.0)

    return eps


In [2]:
for dtype in [np.float16, np.float32, np.float64]:
    computed = find_machine_epsilon(dtype)
    reference = np.finfo(dtype).eps

    print(f"dtype: {dtype.__name__}")
    print(f"computed: {float(computed):.4e}")
    print(f"np.finfo: {float(reference):.4e}")
    print()


dtype: float16
computed: 9.7656e-04
np.finfo: 9.7656e-04

dtype: float32
computed: 1.1921e-07
np.finfo: 1.1921e-07

dtype: float64
computed: 2.2204e-16
np.finfo: 2.2204e-16



## Exercise 2: Catastrophic Cancellation

Use the polynomial `x^2 - 10000.0001x + 1 = 0` to compare the naive quadratic formula with a more stable version based on Vieta's formula.


In [3]:
def quadratic_naive(a, b, c, dtype=np.float64):
    a = dtype(a)
    b = dtype(b)
    c = dtype(c)

    disc = dtype(b * b - dtype(4.0) * a * c)
    sqrt_disc = dtype(np.sqrt(disc))

    x1 = dtype((-b + sqrt_disc) / (dtype(2.0) * a))
    x2 = dtype((-b - sqrt_disc) / (dtype(2.0) * a))

    return x1, x2


def quadratic_stable(a, b, c, dtype=np.float64):
    a = dtype(a)
    b = dtype(b)
    c = dtype(c)

    disc = dtype(b * b - dtype(4.0) * a * c)
    sqrt_disc = dtype(np.sqrt(disc))

    if b < dtype(0.0):
        x1 = dtype((-b + sqrt_disc) / (dtype(2.0) * a))
    else:
        x1 = dtype((-b - sqrt_disc) / (dtype(2.0) * a))

    x2 = dtype(c / (a * x1))

    return x1, x2


def rel_err(approx, true):
    return abs((approx - true) / true)


In [4]:
true_small = np.float64(1.0 / 10000.0001)

for dtype in [np.float32, np.float64]:
    a = dtype(1.0)
    b = dtype(-10000.0001)
    c = dtype(1.0)

    x1_naive, x2_naive = quadratic_naive(a, b, c, dtype)
    x1_stable, x2_stable = quadratic_stable(a, b, c, dtype)

    err_naive = rel_err(dtype(x2_naive), dtype(true_small))
    err_stable = rel_err(dtype(x2_stable), dtype(true_small))

    print(f"dtype: {dtype.__name__}")
    print(f"naive  x1 = {x1_naive}, x2 = {x2_naive}")
    print(f"stable x1 = {x1_stable}, x2 = {x2_stable}")
    print(f"rel.err naive  (small root): {float(err_naive):.4e}")
    print(f"rel.err stable (small root): {float(err_stable):.4e}")
    print()


dtype: float32
naive  x1 = 10000.0, x2 = 0.0
stable x1 = 10000.0, x2 = 9.999999747378752e-05
rel.err naive  (small root): 1.0000e+00
rel.err stable (small root): 0.0000e+00

dtype: float64
naive  x1 = 10000.0, x2 = 0.00010000000020227162
stable x1 = 10000.0, x2 = 0.0001
rel.err naive  (small root): 1.2023e-08
rel.err stable (small root): 1.0000e-08



## Exercise 3: Error Accumulation

Add `0.1` in a loop `n` times and compare the result against the exact value `n * 0.1` for `np.float32` and `np.float64`.


In [5]:
n_values = [10, 100, 1000, 10000, 100000]

for dtype in [np.float32, np.float64]:
    print(f"dtype: {dtype.__name__}")

    for n in n_values:
        total = dtype(0.0)

        for _ in range(n):
            total += dtype(0.1)

        expected = n * 0.1
        rel_error = abs(float(total) - expected) / expected

        print(f"n = {n:6d} | result = {float(total):11.7f} | rel_error = {rel_error:.2e}")

    print()


dtype: float32
n =     10 | result =   1.0000001 | rel_error = 1.19e-07
n =    100 | result =  10.0000019 | rel_error = 1.91e-07
n =   1000 | result =  99.9990463 | rel_error = 9.54e-06
n =  10000 | result = 999.9028931 | rel_error = 9.71e-05
n = 100000 | result = 9998.5566406 | rel_error = 1.44e-04

dtype: float64
n =     10 | result =   1.0000000 | rel_error = 1.11e-16
n =    100 | result =  10.0000000 | rel_error = 1.95e-15
n =   1000 | result = 100.0000000 | rel_error = 1.41e-14
n =  10000 | result = 1000.0000000 | rel_error = 1.59e-13
n = 100000 | result = 10000.0000000 | rel_error = 1.88e-12



## Exercise 4: Extract and Test
For the test, I chose points that can be justified mathematically:

- `c = 0` never escapes, so the function should return `max_iter`.
- `c = 2` gives `z1 = 2` and `z2 = 6`, so with this loop structure the escape count is `2`.

The pytest test is stored in `tests/test_mandelbrot_pixel.py`.


In [6]:
from mandelbrot import mandelbrot_pixel

max_iter = 20
print(f"mandelbrot_pixel(0.0, 0.0, {max_iter}) = {mandelbrot_pixel(0.0, 0.0, max_iter)}")
print(f"mandelbrot_pixel(2.0, 0.0, {max_iter}) = {mandelbrot_pixel(2.0, 0.0, max_iter)}")


mandelbrot_pixel(0.0, 0.0, 20) = 20
mandelbrot_pixel(2.0, 0.0, 20) = 2
